# 🏦 EDA — Home Credit Default Risk
## Análisis Exploratorio de Datos | PySpark + AWS S3

**Proyecto Integrador — MCDA 2026-1 | Universidad EAFIT**  
**Curso:** SI7006 — Almacenamiento y Procesamiento de Grandes Datos  
**Integrantes:** Evelyn Zapata · Julián Valencia · Juan Camilo Villa  

---

### 📋 Objetivo
Realizar un análisis exploratorio completo sobre los datos **raw/bronze** almacenados en AWS S3, sin ninguna limpieza previa, para:
- Entender la estructura, calidad y distribución de cada tabla
- Identificar valores nulos, outliers y desbalances
- Analizar relaciones entre variables y con la variable objetivo `TARGET`
- Almacenar resultados en la zona **refined/silver** del datalake

### 🗂️ Zonas del Datalake
| Zona | S3 Path | Descripción |
|------|---------|-------------|
| Raw/Bronze | `s3://[tu-bucket]/raw/` | Datos originales sin modificar |
| Trusted/Gold | `s3://[tu-bucket]/trusted/` | Datos preparados/limpios |
| Refined/Silver | `s3://[tu-bucket]/refined/` | Resultados EDA y modelos |

### 📁 Tablas del Dataset
1. `application_train.csv` — Tabla principal (TARGET incluido)
2. `bureau.csv` — Créditos previos en otras entidades
3. `bureau_balance.csv` — Balance mensual de créditos bureau
4. `previous_application.csv` — Historial de solicitudes previas
5. `POS_CASH_balance.csv` — Comportamiento créditos consumo
6. `credit_card_balance.csv` — Estado tarjetas de crédito
7. `installments_payments.csv` — Historial de pagos

---
## ⚙️ SECCIÓN 0 — Configuración del Entorno y SparkSession

In [ ]:
# ============================================================
# CELDA 0.1 — Instalación de librerías adicionales
# Ejecutar solo si no están disponibles en el entorno
# ============================================================
import subprocess
import sys

libs = ['matplotlib', 'seaborn', 'pandas', 'numpy']
for lib in libs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '--quiet'])

print('✅ Librerías verificadas correctamente')

In [ ]:
# ============================================================
# CELDA 0.2 — Importaciones globales
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('✅ Importaciones completadas')

In [ ]:
# ============================================================
# CELDA 0.3 — Configuración de paths S3
# ⚠️ AJUSTAR: Reemplaza 'tu-bucket-nombre' con el nombre real
# ============================================================

BUCKET_NAME   = 'tu-bucket-nombre'          # <--- CAMBIAR
RAW_PATH      = f's3://{BUCKET_NAME}/raw/'
TRUSTED_PATH  = f's3://{BUCKET_NAME}/trusted/'
REFINED_PATH  = f's3://{BUCKET_NAME}/refined/eda/'

# Paths individuales de cada tabla
PATHS = {
    'application_train'    : f'{RAW_PATH}application_train.csv',
    'bureau'               : f'{RAW_PATH}bureau.csv',
    'bureau_balance'       : f'{RAW_PATH}bureau_balance.csv',
    'previous_application' : f'{RAW_PATH}previous_application.csv',
    'pos_cash_balance'     : f'{RAW_PATH}POS_CASH_balance.csv',
    'credit_card_balance'  : f'{RAW_PATH}credit_card_balance.csv',
    'installments_payments': f'{RAW_PATH}installments_payments.csv',
}

print('📂 Paths configurados:')
for name, path in PATHS.items():
    print(f'   {name:30s} → {path}')

In [ ]:
# ============================================================
# CELDA 0.4 — Inicialización de SparkSession con S3
# Configurada para AWS EMR o SageMaker Studio con acceso S3
# ============================================================
spark = (
    SparkSession.builder
    .appName('EDA_HomeCreditDefaultRisk')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '8g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

print(f'✅ SparkSession iniciada')
print(f'   Versión Spark : {spark.version}')
print(f'   AppName       : {spark.sparkContext.appName}')
print(f'   Master        : {spark.sparkContext.master}')

In [ ]:
# ============================================================
# CELDA 0.5 — Funciones auxiliares reutilizables para el EDA
# ============================================================

def load_csv(path, table_name):
    """Carga un CSV desde S3 con inferencia de schema."""
    print(f'📥 Cargando: {table_name} ...')
    df = spark.read.csv(path, header=True, inferSchema=True, nullValue='', nanValue='NaN')
    df.createOrReplaceTempView(table_name)
    print(f'   ✅ {df.count():,} filas × {len(df.columns)} columnas')
    return df


def resumen_general(df, nombre):
    """Imprime resumen básico de estructura del dataframe."""
    n_filas = df.count()
    n_cols  = len(df.columns)
    n_num   = sum(1 for f in df.schema.fields if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType)))
    n_cat   = n_cols - n_num
    print(f'\n{'='*55}')
    print(f' 📊 RESUMEN: {nombre}')
    print(f'{'='*55}')
    print(f'  Filas        : {n_filas:>12,}')
    print(f'  Columnas     : {n_cols:>12,}')
    print(f'  Numéricas    : {n_num:>12,}')
    print(f'  Categóricas  : {n_cat:>12,}')
    print(f'{'='*55}')


def analisis_nulos(df, nombre, top_n=30):
    """Calcula nulos absolutos y porcentuales por columna."""
    n = df.count()
    nulos = []
    for col in df.columns:
        cnt = df.filter(F.col(col).isNull()).count()
        nulos.append({'columna': col, 'nulos': cnt, 'porcentaje': round(cnt / n * 100, 2)})
    df_nulos = pd.DataFrame(nulos).sort_values('porcentaje', ascending=False)
    print(f'\n🔍 NULOS — {nombre}')
    print(f'   Columnas con nulos: {(df_nulos["nulos"] > 0).sum()} / {len(df.columns)}')
    return df_nulos


def plot_nulos(df_nulos, titulo, umbral=10):
    """Grafica barras de % de nulos. Resalta las que superan el umbral."""
    df_plot = df_nulos[df_nulos['porcentaje'] > 0].head(30)
    if df_plot.empty:
        print('  ✅ Sin valores nulos en esta tabla.')
        return
    colors = ['#e74c3c' if p > umbral else '#f39c12' for p in df_plot['porcentaje']]
    fig, ax = plt.subplots(figsize=(14, max(4, len(df_plot)*0.35)))
    ax.barh(df_plot['columna'], df_plot['porcentaje'], color=colors)
    ax.axvline(umbral, color='navy', linestyle='--', linewidth=1.5, label=f'Umbral {umbral}%')
    ax.set_xlabel('% de Nulos')
    ax.set_title(f'Porcentaje de Nulos — {titulo}', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'/tmp/nulos_{titulo.lower().replace(" ","_")}.png', dpi=120)
    plt.show()


def estadisticas_spark(df, nombre):
    """Devuelve estadísticas descriptivas como pandas df."""
    num_cols = [f.name for f in df.schema.fields
                if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]
    if not num_cols:
        return None
    stats = df.select(num_cols).describe().toPandas().set_index('summary').T
    stats.columns.name = None
    return stats


def guardar_en_silver(df_pandas, nombre_archivo):
    """Guarda un dataframe pandas como CSV en la zona refined/silver de S3."""
    spark_df = spark.createDataFrame(df_pandas.reset_index().astype(str))
    out_path = f'{REFINED_PATH}{nombre_archivo}'
    spark_df.coalesce(1).write.mode('overwrite').option('header', 'true').csv(out_path)
    print(f'💾 Guardado en Silver: {out_path}')


print('✅ Funciones auxiliares definidas')

---
## 📥 SECCIÓN 1 — Carga de Datos desde S3 (Zona Raw/Bronze)

In [ ]:
# ============================================================
# CELDA 1.1 — Carga de todas las tablas
# ============================================================
df_app      = load_csv(PATHS['application_train'],     'application_train')
df_bureau   = load_csv(PATHS['bureau'],                'bureau')
df_bbal     = load_csv(PATHS['bureau_balance'],        'bureau_balance')
df_prev     = load_csv(PATHS['previous_application'],  'previous_application')
df_pos      = load_csv(PATHS['pos_cash_balance'],      'pos_cash_balance')
df_cc       = load_csv(PATHS['credit_card_balance'],   'credit_card_balance')
df_inst     = load_csv(PATHS['installments_payments'], 'installments_payments')

TABLAS = {
    'application_train'    : df_app,
    'bureau'               : df_bureau,
    'bureau_balance'       : df_bbal,
    'previous_application' : df_prev,
    'pos_cash_balance'     : df_pos,
    'credit_card_balance'  : df_cc,
    'installments_payments': df_inst,
}

print('\n✅ Todas las tablas cargadas correctamente')

In [ ]:
# ============================================================
# CELDA 1.2 — Inventario general del datalake raw
# ============================================================
inventario = []
for nombre, df in TABLAS.items():
    n_filas = df.count()
    n_cols  = len(df.columns)
    n_num   = sum(1 for f in df.schema.fields
                  if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType)))
    inventario.append({
        'Tabla'        : nombre,
        'Filas'        : n_filas,
        'Columnas'     : n_cols,
        'Numéricas'    : n_num,
        'Categóricas'  : n_cols - n_num
    })

df_inventario = pd.DataFrame(inventario)
print('\n📊 INVENTARIO GENERAL DEL DATALAKE RAW')
print(df_inventario.to_string(index=False))

# Gráfico de barras: filas por tabla
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(df_inventario['Tabla'], df_inventario['Filas'] / 1e6,
              color=sns.color_palette('Set2', len(df_inventario)))
ax.set_ylabel('Millones de filas')
ax.set_title('Volumen de datos por tabla (zona Raw/Bronze)', fontsize=13, fontweight='bold')
ax.set_xticklabels(df_inventario['Tabla'], rotation=25, ha='right')
for bar, val in zip(bars, df_inventario['Filas']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

guardar_en_silver(df_inventario, 'inventario_tablas')

---
## 🔬 SECCIÓN 2 — EDA Tabla Principal: application_train

In [ ]:
# ============================================================
# CELDA 2.1 — Estructura y schema
# ============================================================
resumen_general(df_app, 'application_train')
print('\n📋 Schema:')
df_app.printSchema()
print('\n🔎 Primeras 3 filas (selección de columnas clave):')
cols_clave = ['SK_ID_CURR', 'TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL',
              'DAYS_BIRTH', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'EXT_SOURCE_1',
              'EXT_SOURCE_2', 'EXT_SOURCE_3']
df_app.select(cols_clave).show(3, truncate=False)

In [ ]:
# ============================================================
# CELDA 2.2 — Variable objetivo TARGET
# ============================================================
target_dist = df_app.groupBy('TARGET').count().orderBy('TARGET').toPandas()
target_dist['porcentaje'] = (target_dist['count'] / target_dist['count'].sum() * 100).round(2)

print('🎯 DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (TARGET)')
print('   0 = Sin incumplimiento | 1 = Con incumplimiento')
print(target_dist.to_string(index=False))
ratio = target_dist['count'].iloc[0] / target_dist['count'].iloc[1]
print(f'\n   ⚠️  Ratio de desbalance: {ratio:.1f}:1 (mayoría clase 0)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(['No incumplimiento (0)', 'Incumplimiento (1)'],
            target_dist['count'], color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución absoluta de TARGET', fontweight='bold')
axes[0].set_ylabel('Número de registros')
for i, v in enumerate(target_dist['count']):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(target_dist['porcentaje'], labels=['Sin incumplimiento', 'Con incumplimiento'],
            autopct='%1.2f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción de TARGET', fontweight='bold')

plt.suptitle('⚠️ Dataset altamente desbalanceado — Considerar SMOTE o class_weight en modelado',
             fontsize=10, color='darkred', y=1.02)
plt.tight_layout()
plt.show()

guardar_en_silver(target_dist, 'target_distribution')

In [ ]:
# ============================================================
# CELDA 2.3 — Análisis de valores nulos (application_train)
# ============================================================
df_nulos_app = analisis_nulos(df_app, 'application_train')
plot_nulos(df_nulos_app, 'application_train', umbral=40)

# Clasificación de columnas según % de nulos
sin_nulos    = df_nulos_app[df_nulos_app['porcentaje'] == 0]
pocos_nulos  = df_nulos_app[(df_nulos_app['porcentaje'] > 0) & (df_nulos_app['porcentaje'] <= 20)]
muchos_nulos = df_nulos_app[(df_nulos_app['porcentaje'] > 20) & (df_nulos_app['porcentaje'] <= 50)]
criticos     = df_nulos_app[df_nulos_app['porcentaje'] > 50]

print(f'\n📊 Clasificación por nivel de nulos:')
print(f'   ✅ Sin nulos        : {len(sin_nulos):>4} columnas')
print(f'   🟡 1-20% nulos      : {len(pocos_nulos):>4} columnas')
print(f'   🟠 20-50% nulos     : {len(muchos_nulos):>4} columnas')
print(f'   🔴 >50% nulos       : {len(criticos):>4} columnas (considerar eliminar)')

if not criticos.empty:
    print(f'\n   Columnas críticas (>50% nulos):')
    for _, row in criticos.iterrows():
        print(f'      {row["columna"]:40s} → {row["porcentaje"]}%')

guardar_en_silver(df_nulos_app, 'nulos_application_train')

In [ ]:
# ============================================================
# CELDA 2.4 — Estadísticas descriptivas numéricas
# ============================================================
stats_app = estadisticas_spark(df_app, 'application_train')
print('📈 Estadísticas descriptivas (variables numéricas):')
display(stats_app)   # En SageMaker/Databricks
# print(stats_app)   # Alternativa si display no está disponible

guardar_en_silver(stats_app, 'estadisticas_app_train')

In [ ]:
# ============================================================
# CELDA 2.5 — Análisis de variables categóricas clave
# ============================================================
cat_vars = ['CODE_GENDER', 'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE',
            'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
            'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(cat_vars):
    # Calcular distribución con SparkSQL
    dist = (
        df_app.groupBy(col, 'TARGET')
        .count()
        .orderBy(F.desc('count'))
        .toPandas()
    )
    pivot = dist.pivot(index=col, columns='TARGET', values='count').fillna(0)
    pivot.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='white')
    axes[i].set_title(f'{col}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Conteo')
    axes[i].tick_params(axis='x', rotation=35)
    axes[i].legend(['Sin incumplimiento', 'Con incumplimiento'], fontsize=8)

plt.suptitle('Variables Categóricas vs TARGET', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELDA 2.6 — Análisis de variables financieras clave
# ============================================================
# Usamos muestra para graficar eficientemente
sample_app = df_app.sample(False, 0.1, seed=42).toPandas()

fin_vars = [
    ('AMT_INCOME_TOTAL',  'Ingreso Total'),
    ('AMT_CREDIT',        'Monto Crédito'),
    ('AMT_ANNUITY',       'Cuota Anual'),
    ('AMT_GOODS_PRICE',   'Precio del Bien'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (col, label) in enumerate(fin_vars):
    if col not in sample_app.columns:
        continue
    for target_val, color, lbl in [(0, '#2ecc71', 'Sin incumplimiento'),
                                    (1, '#e74c3c', 'Con incumplimiento')]:
        subset = sample_app[sample_app['TARGET'] == target_val][col].dropna()
        # Eliminar outliers extremos para mejor visualización
        q99 = subset.quantile(0.99)
        subset = subset[subset <= q99]
        axes[i].hist(subset, bins=50, alpha=0.6, color=color, label=lbl, density=True)
    axes[i].set_title(f'Distribución: {label}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Densidad')
    axes[i].legend()

plt.suptitle('Distribución de Variables Financieras por TARGET (muestra 10%)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELDA 2.7 — Variables demográficas: Edad e Ingresos
# ============================================================
# DAYS_BIRTH es negativo (días desde nacimiento), convertimos a años
df_age = df_app.withColumn('EDAD_ANOS', (F.abs(F.col('DAYS_BIRTH')) / 365).cast('int'))
df_age_pd = df_age.select('EDAD_ANOS', 'TARGET', 'CODE_GENDER', 'AMT_INCOME_TOTAL').toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de edades
for target_val, color, lbl in [(0, '#2ecc71', 'Sin incumplimiento'),
                                (1, '#e74c3c', 'Con incumplimiento')]:
    subset = df_age_pd[df_age_pd['TARGET'] == target_val]['EDAD_ANOS'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=lbl, density=True)
axes[0].set_title('Distribución de Edades por TARGET', fontweight='bold')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# Edad promedio por grupo TARGET
edad_target = df_age_pd.groupby('TARGET')['EDAD_ANOS'].mean().reset_index()
axes[1].bar(['Sin incumplimiento\n(TARGET=0)', 'Con incumplimiento\n(TARGET=1)'],
            edad_target['EDAD_ANOS'], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Edad Promedio por TARGET', fontweight='bold')
axes[1].set_ylabel('Edad promedio (años)')
for i, v in enumerate(edad_target['EDAD_ANOS']):
    axes[1].text(i, v - 1, f'{v:.1f}', ha='center', fontweight='bold', color='white')

# Género y tasa de incumplimiento
genero_target = (df_age_pd.groupby(['CODE_GENDER', 'TARGET'])
                  .size().reset_index(name='count')
                  .pivot(index='CODE_GENDER', columns='TARGET', values='count'))
genero_target.plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[2].set_title('Género vs TARGET', fontweight='bold')
axes[2].set_xlabel('Género')
axes[2].set_ylabel('Conteo')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print('💡 Insight: Los solicitantes con incumplimiento tienden a ser más jóvenes.')

In [ ]:
# ============================================================
# CELDA 2.8 — Scores externos EXT_SOURCE_1/2/3
# Son las variables más predictivas según literatura del dataset
# ============================================================
ext_vars = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Estadísticas por TARGET con SparkSQL
ext_stats = spark.sql("""
    SELECT 
        TARGET,
        ROUND(AVG(EXT_SOURCE_1), 4) AS avg_ext1,
        ROUND(AVG(EXT_SOURCE_2), 4) AS avg_ext2,
        ROUND(AVG(EXT_SOURCE_3), 4) AS avg_ext3,
        ROUND(STDDEV(EXT_SOURCE_1), 4) AS std_ext1,
        ROUND(STDDEV(EXT_SOURCE_2), 4) AS std_ext2,
        ROUND(STDDEV(EXT_SOURCE_3), 4) AS std_ext3,
        COUNT(*) as n
    FROM application_train
    GROUP BY TARGET
    ORDER BY TARGET
""").toPandas()

print('📊 Estadísticas EXT_SOURCE por TARGET:')
print(ext_stats.to_string(index=False))

# Boxplot de EXT_SOURCEs
ext_pd = df_app.select(ext_vars + ['TARGET']).dropna().sample(False, 0.2, seed=42).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(ext_vars):
    data = [ext_pd[ext_pd['TARGET'] == 0][col].dropna(),
            ext_pd[ext_pd['TARGET'] == 1][col].dropna()]
    bp = axes[i].boxplot(data, labels=['TARGET=0', 'TARGET=1'],
                         patch_artist=True,
                         boxprops=dict(facecolor='lightblue', alpha=0.7))
    bp['boxes'][1].set_facecolor('salmon')
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_ylabel('Score')

plt.suptitle('Scores Externos por TARGET\n(Scores bajos → mayor riesgo de incumplimiento)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

guardar_en_silver(ext_stats, 'ext_source_stats_by_target')

In [ ]:
# ============================================================
# CELDA 2.9 — Detección de Outliers (IQR) con SparkSQL
# ============================================================
vars_outlier = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_EMPLOYED']

print('🔍 DETECCIÓN DE OUTLIERS (método IQR)')
print('='*60)

outlier_summary = []
for col in vars_outlier:
    quantiles = df_app.approxQuantile(col, [0.25, 0.75], 0.01)
    if len(quantiles) < 2:
        continue
    q1, q3 = quantiles
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = df_app.filter(
        (F.col(col) < lower) | (F.col(col) > upper)
    ).count()
    pct = round(n_outliers / df_app.count() * 100, 2)
    outlier_summary.append({'Variable': col, 'Q1': q1, 'Q3': q3, 'IQR': round(iqr, 2),
                             'Lower': round(lower, 2), 'Upper': round(upper, 2),
                             'N_Outliers': n_outliers, 'Pct_Outliers': pct})
    print(f'  {col:25s} → {n_outliers:6,} outliers ({pct}%)')

df_outliers = pd.DataFrame(outlier_summary)
guardar_en_silver(df_outliers, 'outliers_iqr_app_train')

# Caso especial: DAYS_EMPLOYED = 365243 es un código para jubilados/pensionados
n_anomalo = df_app.filter(F.col('DAYS_EMPLOYED') == 365243).count()
print(f'\n  ⚠️  DAYS_EMPLOYED == 365243 (valor anómalo/jubilados): {n_anomalo:,} registros')
print('     Este valor debe ser tratado como categórico o imputado en la preparación')

In [ ]:
# ============================================================
# CELDA 2.10 — Correlación entre variables numéricas y TARGET
# ============================================================
num_cols_corr = [
    'TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    'AMT_GOODS_PRICE'
]
# Filtramos solo columnas que existen
num_cols_corr = [c for c in num_cols_corr if c in df_app.columns]

corr_pd = df_app.select(num_cols_corr).dropna().sample(False, 0.15, seed=42).toPandas()
corr_matrix = corr_pd.corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap completo
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[0], cmap='RdYlGn',
            center=0, annot=True, fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
axes[0].set_title('Matriz de Correlación (variables numéricas clave)', fontweight='bold')

# Correlación específica con TARGET
corr_target = corr_matrix['TARGET'].drop('TARGET').sort_values(key=abs, ascending=False)
colors_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_target]
axes[1].barh(corr_target.index, corr_target.values, color=colors_corr)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlación de Variables con TARGET', fontweight='bold')
axes[1].set_xlabel('Coeficiente de Correlación de Pearson')

plt.tight_layout()
plt.show()

print('\n💡 Top 5 correlaciones con TARGET:')
print(corr_target.head(5).to_string())

guardar_en_silver(corr_target.reset_index().rename(columns={'index':'variable', 'TARGET':'correlacion_target'}),
                  'correlacion_con_target')

In [ ]:
# ============================================================
# CELDA 2.11 — SparkSQL: Análisis de tasas de incumplimiento
# ============================================================

# Tasa de incumplimiento por tipo de educación
tasa_edu = spark.sql("""
    SELECT 
        NAME_EDUCATION_TYPE,
        COUNT(*) as total,
        SUM(TARGET) as incumplimientos,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) as tasa_incumplimiento_pct
    FROM application_train
    GROUP BY NAME_EDUCATION_TYPE
    ORDER BY tasa_incumplimiento_pct DESC
""").toPandas()

print('📊 Tasa de incumplimiento por nivel educativo:')
print(tasa_edu.to_string(index=False))

# Tasa por tipo de ingresos
tasa_income = spark.sql("""
    SELECT 
        NAME_INCOME_TYPE,
        COUNT(*) as total,
        ROUND(SUM(TARGET) * 100.0 / COUNT(*), 2) as tasa_pct
    FROM application_train
    GROUP BY NAME_INCOME_TYPE
    ORDER BY tasa_pct DESC
""").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(tasa_edu['NAME_EDUCATION_TYPE'], tasa_edu['tasa_incumplimiento_pct'],
             color=sns.color_palette('Reds_r', len(tasa_edu)))
axes[0].set_title('Tasa de Incumplimiento por Educación', fontweight='bold')
axes[0].set_xlabel('% Incumplimiento')

axes[1].barh(tasa_income['NAME_INCOME_TYPE'], tasa_income['tasa_pct'],
             color=sns.color_palette('Oranges_r', len(tasa_income)))
axes[1].set_title('Tasa de Incumplimiento por Tipo de Ingreso', fontweight='bold')
axes[1].set_xlabel('% Incumplimiento')

plt.tight_layout()
plt.show()

guardar_en_silver(tasa_edu, 'tasa_incumplimiento_por_educacion')
guardar_en_silver(tasa_income, 'tasa_incumplimiento_por_ingreso')

---
## 🔬 SECCIÓN 3 — EDA Tabla: bureau (Historial Crediticio Externo)

In [ ]:
# ============================================================
# CELDA 3.1 — Estructura y nulos de bureau
# ============================================================
resumen_general(df_bureau, 'bureau')
df_bureau.select(['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE',
                  'CREDIT_TYPE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT']).show(5)

df_nulos_bureau = analisis_nulos(df_bureau, 'bureau')
plot_nulos(df_nulos_bureau, 'bureau')
guardar_en_silver(df_nulos_bureau, 'nulos_bureau')

In [ ]:
# ============================================================
# CELDA 3.2 — Análisis de créditos activos y morosos
# ============================================================
credit_active = spark.sql("""
    SELECT CREDIT_ACTIVE, COUNT(*) as total,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bureau), 2) as pct
    FROM bureau
    GROUP BY CREDIT_ACTIVE
    ORDER BY total DESC
""").toPandas()

credit_type = spark.sql("""
    SELECT CREDIT_TYPE, COUNT(*) as total
    FROM bureau
    GROUP BY CREDIT_TYPE
    ORDER BY total DESC
    LIMIT 10
""").toPandas()

print('Estado de créditos bureau:')
print(credit_active.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(credit_active['CREDIT_ACTIVE'], credit_active['pct'],
            color=sns.color_palette('Set2', len(credit_active)))
axes[0].set_title('Estado de Créditos Bureau', fontweight='bold')
axes[0].set_ylabel('% del total')

axes[1].barh(credit_type['CREDIT_TYPE'], credit_type['total'],
             color=sns.color_palette('Set3', len(credit_type)))
axes[1].set_title('Top 10 Tipos de Crédito Bureau', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELDA 3.3 — Agregación bureau por cliente y join con TARGET
# ============================================================
bureau_agg = spark.sql("""
    SELECT 
        b.SK_ID_CURR,
        COUNT(b.SK_ID_BUREAU)                         AS num_creditos_bureau,
        SUM(CASE WHEN b.CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS creditos_activos,
        SUM(CASE WHEN b.CREDIT_ACTIVE = 'Bad debt' THEN 1 ELSE 0 END) AS deudas_malas,
        ROUND(AVG(b.AMT_CREDIT_SUM), 2)               AS avg_monto_credito,
        ROUND(SUM(b.AMT_CREDIT_SUM_DEBT), 2)          AS deuda_total,
        ROUND(AVG(b.DAYS_CREDIT), 2)                  AS avg_dias_credito,
        MAX(b.DAYS_CREDIT_ENDDATE)                    AS ultimo_vencimiento
    FROM bureau b
    GROUP BY b.SK_ID_CURR
""").toPandas()

# Join con TARGET
target_pd = df_app.select('SK_ID_CURR', 'TARGET').toPandas()
bureau_target = bureau_agg.merge(target_pd, on='SK_ID_CURR', how='inner')

print(f'📊 Clientes con historial bureau: {len(bureau_agg):,}')
print(f'   Con coincidencia en application_train: {len(bureau_target):,}')
print(f'\nPromedio de créditos bureau por TARGET:')
print(bureau_target.groupby('TARGET')[['num_creditos_bureau', 'creditos_activos', 'deudas_malas']].mean())

guardar_en_silver(bureau_agg, 'bureau_features_por_cliente')

---
## 🔬 SECCIÓN 4 — EDA Tabla: bureau_balance

In [ ]:
# ============================================================
# CELDA 4.1 — Estructura y estados de bureau_balance
# ============================================================
resumen_general(df_bbal, 'bureau_balance')
df_bbal.show(5)

# Distribución de STATUS (C=cerrado, 0-5=mora, X=sin info)
status_dist = spark.sql("""
    SELECT STATUS, COUNT(*) as total,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bureau_balance), 2) as pct
    FROM bureau_balance
    GROUP BY STATUS
    ORDER BY STATUS
""").toPandas()

print('\n📊 Distribución de STATUS en bureau_balance:')
print('   C=Cerrado, 0=Sin mora, 1-5=Meses en mora, X=Sin información')
print(status_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(status_dist['STATUS'], status_dist['pct'],
       color=sns.color_palette('RdYlGn_r', len(status_dist)))
ax.set_title('Distribución de STATUS en bureau_balance\n(0=Sin mora → 5=Mora severa)',
             fontweight='bold')
ax.set_ylabel('% del total')
plt.tight_layout()
plt.show()

---
## 🔬 SECCIÓN 5 — EDA Tabla: previous_application

In [ ]:
# ============================================================
# CELDA 5.1 — Estructura y análisis de solicitudes previas
# ============================================================
resumen_general(df_prev, 'previous_application')
df_nulos_prev = analisis_nulos(df_prev, 'previous_application')
plot_nulos(df_nulos_prev, 'previous_application')

# Decisiones de las solicitudes previas
decision_dist = spark.sql("""
    SELECT NAME_CONTRACT_STATUS, COUNT(*) as total,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM previous_application), 2) as pct
    FROM previous_application
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY total DESC
""").toPandas()

print('📊 Decisiones sobre solicitudes previas:')
print(decision_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.pie(decision_dist['pct'], labels=decision_dist['NAME_CONTRACT_STATUS'],
       autopct='%1.1f%%', colors=sns.color_palette('Set2', len(decision_dist)),
       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Estado de Contratos en Solicitudes Previas', fontweight='bold')
plt.tight_layout()
plt.show()

guardar_en_silver(df_nulos_prev, 'nulos_previous_application')

In [ ]:
# ============================================================
# CELDA 5.2 — Agregación de solicitudes previas por cliente
# ============================================================
prev_agg = spark.sql("""
    SELECT 
        SK_ID_CURR,
        COUNT(SK_ID_PREV)                                                        AS num_solicitudes_prev,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)      AS aprobadas,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END)      AS rechazadas,
        ROUND(AVG(AMT_APPLICATION), 2)                                           AS avg_monto_solicitado,
        ROUND(AVG(AMT_CREDIT), 2)                                                AS avg_credito_aprobado,
        ROUND(AVG(DAYS_DECISION), 2)                                             AS avg_dias_decision
    FROM previous_application
    GROUP BY SK_ID_CURR
""").toPandas()

print(f'📊 Clientes con solicitudes previas: {len(prev_agg):,}')
print(f'\nEstadísticas:')
print(prev_agg[['num_solicitudes_prev', 'aprobadas', 'rechazadas']].describe().round(2))

guardar_en_silver(prev_agg, 'previous_app_features_por_cliente')

---
## 🔬 SECCIÓN 6 — EDA Tablas: POS_CASH, credit_card, installments

In [ ]:
# ============================================================
# CELDA 6.1 — POS_CASH_balance: comportamiento mensual
# ============================================================
resumen_general(df_pos, 'pos_cash_balance')
df_pos.show(5)

# Estado del contrato POS
pos_status = spark.sql("""
    SELECT NAME_CONTRACT_STATUS, COUNT(*) as total
    FROM pos_cash_balance
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY total DESC
""").toPandas()
print('\nEstados POS_CASH:')
print(pos_status.to_string(index=False))

# Cuotas pendientes vs pagadas
pos_cuotas = spark.sql("""
    SELECT 
        SK_ID_CURR,
        AVG(CNT_INSTALMENT)          AS avg_cuotas_totales,
        AVG(CNT_INSTALMENT_FUTURE)   AS avg_cuotas_pendientes,
        COUNT(*)                     AS meses_registrados
    FROM pos_cash_balance
    GROUP BY SK_ID_CURR
""").toPandas()

guardar_en_silver(pos_cuotas, 'pos_cash_features_por_cliente')
print(f'\n📊 Clientes con historial POS: {len(pos_cuotas):,}')

In [ ]:
# ============================================================
# CELDA 6.2 — credit_card_balance
# ============================================================
resumen_general(df_cc, 'credit_card_balance')
df_nulos_cc = analisis_nulos(df_cc, 'credit_card_balance')
plot_nulos(df_nulos_cc, 'credit_card_balance')

# Utilización de tarjeta por cliente
cc_agg = spark.sql("""
    SELECT 
        SK_ID_CURR,
        AVG(AMT_BALANCE)                AS avg_balance_tc,
        AVG(AMT_CREDIT_LIMIT_ACTUAL)    AS avg_limite_credito,
        AVG(AMT_DRAWINGS_CURRENT)       AS avg_retiros,
        AVG(SK_DPD)                     AS avg_dias_mora,
        MAX(SK_DPD)                     AS max_dias_mora
    FROM credit_card_balance
    GROUP BY SK_ID_CURR
""").toPandas()

cc_agg['utilizacion_pct'] = (cc_agg['avg_balance_tc'] / cc_agg['avg_limite_credito'] * 100).clip(0, 200)

print(f'📊 Clientes con historial de tarjeta: {len(cc_agg):,}')
print(f'   Utilización promedio: {cc_agg["utilizacion_pct"].mean():.1f}%')
print(f'   Días de mora promedio: {cc_agg["avg_dias_mora"].mean():.2f}')

guardar_en_silver(cc_agg, 'credit_card_features_por_cliente')

In [ ]:
# ============================================================
# CELDA 6.3 — installments_payments: comportamiento de pago
# ============================================================
resumen_general(df_inst, 'installments_payments')
df_inst.show(5)

# Análisis de puntualidad en pagos
inst_agg = spark.sql("""
    SELECT 
        SK_ID_CURR,
        COUNT(*)                                                           AS n_pagos,
        ROUND(AVG(AMT_INSTALMENT), 2)                                     AS avg_cuota,
        ROUND(AVG(AMT_PAYMENT), 2)                                        AS avg_pago,
        ROUND(AVG(AMT_PAYMENT - AMT_INSTALMENT), 2)                      AS diferencia_pago,
        ROUND(AVG(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT), 2)              AS dias_retraso_avg,
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END) AS pagos_tardios
    FROM installments_payments
    GROUP BY SK_ID_CURR
""").toPandas()

inst_agg['pct_pagos_tardios'] = (inst_agg['pagos_tardios'] / inst_agg['n_pagos'] * 100).round(2)

print(f'📊 Estadísticas de comportamiento de pago:')
print(inst_agg[['dias_retraso_avg', 'pct_pagos_tardios', 'diferencia_pago']].describe().round(2))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(inst_agg['dias_retraso_avg'].clip(-30, 60), bins=50, color='#3498db', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='A tiempo')
axes[0].set_title('Distribución de Días de Retraso en Pagos', fontweight='bold')
axes[0].set_xlabel('Días (negativo=anticipado, positivo=tardío)')
axes[0].legend()

axes[1].hist(inst_agg['pct_pagos_tardios'].clip(0, 100), bins=40, color='#e67e22', edgecolor='white')
axes[1].set_title('% de Pagos Tardíos por Cliente', fontweight='bold')
axes[1].set_xlabel('% pagos tardíos')

plt.tight_layout()
plt.show()

guardar_en_silver(inst_agg, 'installments_features_por_cliente')

---
## 🔗 SECCIÓN 7 — Integración de Tablas: Dataset Analítico Consolidado

In [ ]:
# ============================================================
# CELDA 7.1 — Join de todas las features agregadas
# ============================================================

# Convertir aggregaciones a Spark DataFrames para hacer el join distribuido
bureau_agg_sp  = spark.createDataFrame(bureau_agg)
prev_agg_sp    = spark.createDataFrame(prev_agg)
pos_cuotas_sp  = spark.createDataFrame(pos_cuotas)
cc_agg_sp      = spark.createDataFrame(cc_agg)
inst_agg_sp    = spark.createDataFrame(inst_agg)

# Join progresivo sobre SK_ID_CURR
df_master = (
    df_app
    .join(bureau_agg_sp,  on='SK_ID_CURR', how='left')
    .join(prev_agg_sp,    on='SK_ID_CURR', how='left')
    .join(pos_cuotas_sp,  on='SK_ID_CURR', how='left')
    .join(cc_agg_sp,      on='SK_ID_CURR', how='left')
    .join(inst_agg_sp,    on='SK_ID_CURR', how='left')
)

n_total = df_master.count()
n_cols  = len(df_master.columns)
print(f'✅ Dataset maestro consolidado:')
print(f'   Filas    : {n_total:,}')
print(f'   Columnas : {n_cols}')

# Guardar en zona Silver como Parquet (más eficiente)
master_silver_path = f'{REFINED_PATH}dataset_maestro_eda/'
df_master.write.mode('overwrite').parquet(master_silver_path)
print(f'\n💾 Dataset maestro guardado en Silver: {master_silver_path}')

In [ ]:
# ============================================================
# CELDA 7.2 — Análisis cruzado: features externas vs TARGET
# ============================================================
df_master.createOrReplaceTempView('dataset_maestro')

analisis_cruzado = spark.sql("""
    SELECT 
        TARGET,
        ROUND(AVG(num_creditos_bureau), 2)    AS avg_creditos_bureau,
        ROUND(AVG(creditos_activos), 2)       AS avg_creditos_activos,
        ROUND(AVG(deudas_malas), 2)           AS avg_deudas_malas,
        ROUND(AVG(num_solicitudes_prev), 2)   AS avg_solicitudes_previas,
        ROUND(AVG(aprobadas), 2)              AS avg_aprobadas,
        ROUND(AVG(rechazadas), 2)             AS avg_rechazadas,
        ROUND(AVG(dias_retraso_avg), 2)       AS avg_dias_retraso,
        ROUND(AVG(pct_pagos_tardios), 2)      AS avg_pct_tardios,
        ROUND(AVG(avg_dias_mora), 2)          AS avg_dias_mora_tc,
        COUNT(*) AS n
    FROM dataset_maestro
    GROUP BY TARGET
    ORDER BY TARGET
""").toPandas()

print('📊 ANÁLISIS CRUZADO: Comportamiento promedio por TARGET')
print(analisis_cruzado.T.to_string())

guardar_en_silver(analisis_cruzado, 'analisis_cruzado_por_target')

---
## 📊 SECCIÓN 8 — Dashboard Final de Hallazgos

In [ ]:
# ============================================================
# CELDA 8.1 — Dashboard visual de hallazgos clave del EDA
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- Panel 1: Desbalance TARGET ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.pie(target_dist['porcentaje'],
        labels=['Sin incumplimiento\n(91.9%)', 'Con incumplimiento\n(8.1%)'],
        colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Desbalance de TARGET', fontweight='bold', fontsize=11)

# --- Panel 2: EXT_SOURCE medias ---
ax2 = fig.add_subplot(gs[0, 1])
ext_means = ext_stats[['avg_ext1', 'avg_ext2', 'avg_ext3']].set_index(ext_stats['TARGET'])
ext_means.T.plot(kind='bar', ax=ax2, color=['#2ecc71', '#e74c3c'], width=0.6)
ax2.set_title('EXT_SOURCE Promedio por TARGET', fontweight='bold', fontsize=11)
ax2.set_xticklabels(['EXT_1', 'EXT_2', 'EXT_3'], rotation=0)
ax2.legend(['TARGET=0', 'TARGET=1'], fontsize=9)
ax2.set_ylabel('Score promedio')

# --- Panel 3: Volumen tablas ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.barh(df_inventario['Tabla'],
         df_inventario['Filas'] / 1e6,
         color=sns.color_palette('Set2', len(df_inventario)))
ax3.set_title('Volumen por Tabla (M filas)', fontweight='bold', fontsize=11)
ax3.set_xlabel('Millones de filas')

# --- Panel 4: Tasa incumplimiento por educación ---
ax4 = fig.add_subplot(gs[1, :])
colors_edu = sns.color_palette('Reds_r', len(tasa_edu))
ax4.bar(tasa_edu['NAME_EDUCATION_TYPE'], tasa_edu['tasa_incumplimiento_pct'], color=colors_edu)
ax4.set_title('Tasa de Incumplimiento por Nivel Educativo', fontweight='bold', fontsize=12)
ax4.set_ylabel('% Incumplimiento')
for i, v in enumerate(tasa_edu['tasa_incumplimiento_pct']):
    ax4.text(i, v + 0.1, f'{v}%', ha='center', fontsize=9)

# --- Panel 5: Análisis cruzado ---
ax5 = fig.add_subplot(gs[2, :])
metricas = ['avg_creditos_bureau', 'avg_deudas_malas', 'avg_solicitudes_previas',
            'avg_rechazadas', 'avg_pct_tardios']
metricas_existentes = [m for m in metricas if m in analisis_cruzado.columns]
x = np.arange(len(metricas_existentes))
width = 0.35
vals_0 = analisis_cruzado[analisis_cruzado['TARGET'] == 0][metricas_existentes].values[0]
vals_1 = analisis_cruzado[analisis_cruzado['TARGET'] == 1][metricas_existentes].values[0]
ax5.bar(x - width/2, vals_0, width, color='#2ecc71', label='Sin incumplimiento (0)')
ax5.bar(x + width/2, vals_1, width, color='#e74c3c', label='Con incumplimiento (1)')
ax5.set_xticks(x)
ax5.set_xticklabels(['Créditos\nBureau', 'Deudas\nMalas', 'Solicitudes\nPrevias',
                     'Rechazadas', 'Pagos\nTardíos%'], fontsize=10)
ax5.set_title('Comportamiento Promedio por TARGET (Features Derivadas)', fontweight='bold', fontsize=12)
ax5.legend()

fig.suptitle('Dashboard EDA — Home Credit Default Risk\nUniversidad EAFIT | 2026-1',
             fontsize=15, fontweight='bold', y=1.02)
plt.savefig('/tmp/dashboard_eda_final.png', dpi=150, bbox_inches='tight')
plt.show()

# Subir imagen a S3 Silver
import boto3
s3 = boto3.client('s3')
s3.upload_file('/tmp/dashboard_eda_final.png', BUCKET_NAME, 'refined/eda/dashboard_eda_final.png')
print('💾 Dashboard guardado en S3 refined/eda/dashboard_eda_final.png')

---
## 📝 SECCIÓN 9 — Hallazgos y Conclusiones del EDA

In [ ]:
# ============================================================
# CELDA 9.1 — Resumen ejecutivo de hallazgos
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         RESUMEN DE HALLAZGOS — EDA Home Credit Default Risk          ║
║               Universidad EAFIT | MCDA 2026-1                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. DESBALANCE DE CLASES                                             ║
║     • TARGET=0 (sin incumplimiento): ~91.9%                          ║
║     • TARGET=1 (con incumplimiento): ~8.1%                           ║
║     → ACCIÓN: Aplicar SMOTE, class_weight o umbral personalizado     ║
║                                                                      ║
║  2. VALORES NULOS                                                    ║
║     • application_train: ~65 columnas con nulos (hasta 70%+)         ║
║     • EXT_SOURCE_1 tiene alta tasa de nulos (~56%)                   ║
║     • DAYS_EMPLOYED=365243 es valor anómalo (jubilados)              ║
║     → ACCIÓN: Imputar con mediana/moda o marcar como categoría       ║
║                                                                      ║
║  3. VARIABLES MÁS DISCRIMINANTES                                     ║
║     • EXT_SOURCE_2 y EXT_SOURCE_3: mayor correlación negativa        ║
║     • Scores externos bajos → mayor probabilidad de incumplimiento   ║
║     • Solicitantes más jóvenes tienen mayor tasa de incumplimiento   ║
║                                                                      ║
║  4. COMPORTAMIENTO CREDITICIO PREVIO                                 ║
║     • Clientes con deudas_malas > 0 tienen mayor riesgo             ║
║     • Mayor % de pagos tardíos correlaciona con incumplimiento       ║
║     • Solicitudes rechazadas previamente → señal de riesgo           ║
║                                                                      ║
║  5. VARIABLES FINANCIERAS                                            ║
║     • Ratio AMT_ANNUITY / AMT_INCOME_TOTAL es indicador clave        ║
║     • Montos muy altos o ingresos muy bajos → mayor riesgo           ║
║                                                                      ║
║  6. OUTLIERS DETECTADOS                                              ║
║     • AMT_INCOME_TOTAL: valores extremos (>99 percentil)             ║
║     • DAYS_EMPLOYED: valor anómalo 365243 (~8% de registros)         ║
║     → ACCIÓN: Winsorizar al percentil 99 o transformar log           ║
║                                                                      ║
║  7. PRÓXIMOS PASOS (Preparación de datos)                            ║
║     • Imputación de nulos                                            ║
║     • Codificación de variables categóricas (OneHot / Label)         ║
║     • Feature Engineering: ratios financieros, features temporales   ║
║     • Selección de features por correlación e importancia            ║
║     • Construcción del dataset final para modelado SparkML           ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# CELDA 9.2 — Inventario de archivos guardados en Silver
# ============================================================
import boto3

s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket(BUCKET_NAME)

print(f'📂 Archivos guardados en zona refined/silver — EDA:')
print(f'   Ruta: s3://{BUCKET_NAME}/refined/eda/')
print()
for obj in sorted(bucket.objects.filter(Prefix='refined/eda/'), key=lambda x: x.key):
    size_kb = obj.size / 1024
    print(f'   📄 {obj.key.split("/")[-1]:50s} ({size_kb:.1f} KB)')

print()
print('✅ EDA completado y resultados almacenados en Silver/Refined.')
print('   Listo para la fase de preparación de datos → zona Trusted/Gold')

In [ ]:
# ============================================================
# CELDA FINAL — Cierre de SparkSession
# ============================================================
spark.stop()
print('🔴 SparkSession cerrada. EDA finalizado.')